# 50 — Runtime & Model Registry

Congela el contrato operacional de ARGOS V1: runtime, hashes, cards, registry y estado inicial. M6 inicia en `candidate/shadow`, nunca en promoción automática a producción.

In [1]:
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


{'m6_schema': 'm6-production-v1',
 'model_version': 'argos-v1',
 'operating_mode': 'shadow',
 'scoring_mode': 'batch',
 'human_review_required': True,
 'production_promotion_authorized': False}

In [2]:
from argos.production.registry import BUILD_IMPLEMENTATION, build_model_registry

print("implementation:", BUILD_IMPLEMENTATION)
result = build_model_registry(
    root=paths.root,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.validation

implementation: m6-50-runtime-registry-v1


,check_id,category,blocking,passed,observed,requirement
0,m5_gate_present,prerequisite,True,True,"{""path"": ""/Users/macm1pro/Desktop/projects/arg...",M6 starts only after the computable M5 gate is...
1,frozen_model_hashes_match_selection,lineage,True,True,"{""calibrator"": ""95e8e506f92b191f7315a1f29860bf...",registry points to the exact frozen ARGOS V1 m...
2,runtime_matches_lock,runtime,True,True,"{""expected_python"": ""3.12"", ""expected_sklearn""...",serialized model runtime is pinned and matches...
3,governance_docs_hashed,governance,True,True,"{""model_card"": ""/Users/macm1pro/Desktop/projec...","model, robustness and monitoring documentation..."
4,production_not_pre_authorized,governance,True,True,"{""allow_production_promotion"": false, ""human_r...","M6 package starts batch-first, shadow-only and..."


In [3]:
blocking = result.validation.loc[result.validation["blocking"]]
print("50 ✅ PASS" if result.ready_for_next_notebook else "50 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Registry:", result.registry_record)
print("Deployment state:", result.deployment_state)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook

50 ✅ PASS
Blocking checks: 5/5
Registry: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/registry/argos_v1.json
Deployment state: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/registry/deployment_state.json
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m6_runtime_registry.json
